# Agente de carga académica v3: demostración paso a paso

Sistema Multiagente (Historial, Carga, Horarios y Manager) coordinado con **Contract Net**, auditado con **Shapley** y con una **ontología OWL** como base de conocimiento.

> **Datos:** el plan de estudios es el **real** de Ingeniería en Sistemas Computacionales de la Ibero Puebla (79 materias exportadas de Intrauia, con su seriación). La oferta de grupos, los profesores y los alumnos son **sintéticos**.

> Kernel: usa el entorno `.venv` de esta carpeta (Python 3.12).
> El LLM es opcional. Si quieres usarlo, ejecuta antes `./iniciar_ollama_cpu.sh` en una terminal: así Ollama corre **solo en CPU** y no truena con ROCm.

In [ ]:
from carga.kb import BaseConocimiento
from carga.llm import ClienteLLM, diagnostico
from carga import sistema

print("\n".join(diagnostico()))
USAR_LLM = False          # cámbialo a True si levantaste ./iniciar_ollama_cpu.sh
kb = BaseConocimiento()   # carga la ontología poblada UNA sola vez
llm = ClienteLLM(habilitado=USAR_LLM)

## 1. Base de conocimiento: el alumno según la ontología (consultado por SPARQL)

In [ ]:
est = kb.estudiante("A001")
print(est.nombre, "| semestre", est.semestre, "| promedio", est.promedio)
print("Aprobadas:", ", ".join(est.aprobadas))
print("Reprobadas:", est.reprobadas)
print("Preferencias guardadas:", est.preferencias)

## 2. AgenteHorarios interpreta la petición en lenguaje natural

In [ ]:
peticion = "Me interesa inteligencia artificial, prefiero la tarde y no quiero clases los viernes"
s = sistema.nueva_sesion("A001", peticion, usar_llm=USAR_LLM, eco=True, kb=kb, llm=llm)
from carga.preferencias import describir
print(describir(s.prefs), "\n→", s.prefs["interpretado_por"])

## 3. Contract Net: cfp → propose → accept/reject

Desde la mejora 1 compiten **3 AgentesCarga** (CargaAvance, CargaEquilibrio y CargaPreferencias). Cada uno oferta sus 2 mejores cargas según su estrategia y el Manager las califica con la función oficial.

In [ ]:
res = sistema.correr_contract_net(s)
for p in res.propuestas:
    print(p.id, p.autor, p.puntuacion, p.materias, p.criterios)
print("\nGanador del contrato:", res.ganadora.autor)

## 4. Transparencia: valores de Shapley (exactos, 8 coaliciones)

In [ ]:
from carga.reporte import _coal
phi = sistema.correr_shapley(s)
for S, v in sorted(s.v.items(), key=lambda x: len(x[0])):
    print(f"v({_coal(S)}) = {v}")
print("\nφ:", phi, "\nsuma:", round(sum(phi.values()), 2), "= v(N)")

### 4.1 Shapley entre los 3 contratistas

Segundo juego: Historial y Horarios siempre están; juegan solo los AgentesCarga. v(S) se mide contra la carga ingenua, para que v(∅) = 0. Si dos contratistas ofertan lo mismo, Shapley les paga lo mismo (**simetría**).

In [ ]:
from carga.reporte import _coal_c, licitacion
for c, n, mejor, auto, gano in licitacion(res):
    print(f"{c:18} ofertas={n}  mejor oficial={mejor}  autoevaluación={auto}  {'GANA' if gano else ''}")
print()
for S, v in sorted(s.v_c.items(), key=lambda x: len(x[0])):
    print(f"v({_coal_c(S)}) = {v}")
print("\nφ contratistas:", s.phi_c, "\nsuma:", round(sum(s.phi_c.values()), 2), "= v(N)")

## 5. Razonador HermiT sobre las clases definidas

In [ ]:
for clase, miembros in kb.inferencias_hermit().items():
    print(f"{clase:20} {miembros}")

## 6. Recomendación final, reporte y resultados escritos de vuelta en la ontología

In [ ]:
rutas = sistema.cerrar(s)
from IPython.display import Markdown, display
display(Markdown(rutas["reporte"].read_text(encoding="utf-8")))